# M3 Agentic AI - 将函数变成工具

## 1. 引言

### 1.1. 实验概览

在本次无评分实验中，你将使用智谱AI SDK 创建一组工具并提供给 LLM。你将看到 LLM 如何请求使用工具，以及在任务相关时如何选择合适的工具。

### 🎯 1.2 学习目标

将工具调用的设计模式应用到智能体工作流。

为实现这一点，你将通过智谱AI SDK 向 LLM 提供受控的 Python 函数访问能力，管理参数传递与执行流程，并验证通过工具编排生成的多步输出。

## 2. 环境设置：初始化环境与客户端

与之前实验一样，你将从初始化环境开始。现在导入若干包，并在后续构建工具时根据需要继续导入。

In [1]:
import json
import os
from datetime import datetime
from dotenv import load_dotenv

_ = load_dotenv()

### 2.1 开始使用智谱AI SDK

接下来，初始化你在之前课程中见到的 **智谱AI** 客户端。初始化后，它将作为你生成智能体响应与调用工具的接口。

运行下方单元以初始化客户端。

In [ ]:
# 导入 utils.py 中的 zai_client
import sys
from utils import *

# 使用已有的客户端实例
client = zai_client

## 3. 构建你的第一个工具

### 3.1 定义你的函数

环境已就绪，现在该创建你的第一个工具了。运行下方单元，定义一个返回当前时间字符串的函数。注意该工具包含了函数用途的 docstring 说明，这对智谱AI 很重要，因为它会用这些信息帮助 LLM 了解该工具。

In [3]:
def get_current_time():
    """
    返回当前时间的字符串。
    """
    return datetime.now().strftime("%H:%M:%S")

# 测试你的函数以查看其具体返回值
get_current_time()

'16:11:15'

<div style="background-color:#ffe4e1; padding:12px; border-radius:6px; color:black;">
  <strong>注意：</strong> DeepLearning.AI 平台默认使用格林尼治标准时间（GMT）。如果在本地运行，该函数将返回你的本地时间。
</div>

### 3.2 将函数变成 LLM 工具

现在我们用智谱AI SDK 将该工具提供给 LLM 并获取响应。要设置你的工具，首先需要设置来自 LLM 的 `response`。创建响应前需先构建消息结构：包含用户的提示词，以及表示会话历史的字典，其中每条消息包含 `role`（例如 "user"、"assistant"、"system"）与 `content`。

In [4]:
# 消息结构
prompt = "现在几点？"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

定义好消息结构后，你可以构建聊天补全（chat completion）。这将调用 LLM 并返回结果。我们来看一下该调用的参数：
* `model`：所用的模型 (glm-4.7-flash)
* `messages`：传递给 LLM 的消息列表
* `tools`：LLM 可使用的工具列表（以 JSON Schema 格式定义）

智谱AI SDK 与 AISuite 的不同之处在于，智谱AI SDK 需要手动管理工具调用的完整流程，不会自动处理 max_turns。

In [5]:
# 定义工具描述（JSON Schema格式）
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "返回当前时间的字符串",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

# 调用模型
response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=tools
)

# 处理工具调用
def handle_tool_calls(response, messages, client, tools, max_iterations=5):
    """处理工具调用的循环，直到不再需要调用工具或达到最大迭代次数"""
    for iteration in range(max_iterations):
        if response.choices[0].message.tool_calls:
            # LLM请求调用工具
            tool_call = response.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            
            # 解析参数
            if tool_call.function.arguments:
                args = json.loads(tool_call.function.arguments)
            else:
                args = {}
            
            # 执行对应的工具函数
            if tool_name == "get_current_time":
                result = get_current_time()
                print(f"工具执行结果: {result}")
            else:
                result = f"未知工具: {tool_name}"
            
            # 将工具结果返回给LLM继续处理
            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": [tool_call.model_dump()]
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })
            
            # 获取最终响应
            response = client.chat.completions.create(
                model="glm-4.7-flash",
                messages=messages,
                tools=tools
            )
        else:
            # LLM直接响应，没有调用工具
            break
    return response

# 处理工具调用并获取最终响应
final_response = handle_tool_calls(response, messages, client, tools, max_iterations=5)

# 查看 LLM 的响应
print_html(f"最终响应: {final_response.choices[0].message.content}")

工具执行结果: 16:11:31


至此，你已让 LLM 获得了工具访问能力！智谱AI SDK 将你的函数转换为工具，增强了 LLM 关于世界的可用知识。

### 3.3 更深入地查看响应
虽然最终响应的内容符合预期，但 `response` 背后实际上发生了许多事情。我们可以查看原始响应来理解工具调用的流程。

In [6]:
# 查看原始响应结构
print("原始响应:")
print(json.dumps(response.model_dump(), indent=2, default=str))

原始响应:
{
  "model": "glm-4.7-flash",
  "created": 1772093491,
  "choices": [
    {
      "index": 0,
      "finish_reason": "tool_calls",
      "message": {
        "content": "",
        "role": "assistant",
        "reasoning_content": "\u7528\u6237\u7528\u4e2d\u6587\u95ee\"\u73b0\u5728\u51e0\u70b9\uff1f\"\uff0c\u610f\u601d\u662f\"what time is it now?\"\u3002\u6211\u6709\u4e00\u4e2aget_current_time\u51fd\u6570\u53ef\u4ee5\u83b7\u53d6\u5f53\u524d\u65f6\u95f4\uff0c\u6211\u5e94\u8be5\u8c03\u7528\u5b83\u6765\u56de\u7b54\u7528\u6237\u7684\u95ee\u9898\u3002",
        "tool_calls": [
          {
            "id": "call_-7851660534502913817",
            "function": {
              "arguments": "{}",
              "name": "get_current_time"
            },
            "type": "function",
            "index": 0
          }
        ]
      }
    }
  ],
  "request_id": "20260226161127140a90a9ccc742bc",
  "id": "20260226161127140a90a9ccc742bc",
  "usage": {
    "prompt_tokens": 144,
    "prompt_to

如你所见，LLM 发送了一条消息请求使用 `get_current_time`。该调用在你的机器上执行后返回给 LLM。最终，LLM 基于完整的会话历史生成了最终回复。与 AISuite 不同，智谱AI SDK 需要手动处理从响应中提取工具调用消息、在本地执行并将结果回传给 LLM 的所有复杂细节。

### 3.4 手动定义工具

你已经看到工具的 docstring 可以帮助自动把函数变成 LLM 可用的工具。这非常方便。但在幕后究竟发生了什么，才能让你的函数被包装为工具？

实际上，LLM 所看到的工具结构更为复杂。下面我们看看 LLM *实际* 获得的工具是怎样的。与之前一样，工具通过列表传入；但列表中的每个工具都遵循一套约定的 schema。该 schema 包含几个关键部分：
* `name`：你在本地定义的对应函数名
* `description`：解释函数用途的描述，供 LLM 判断何时使用
* `parameters`：若函数有参数，这里也会描述参数名与参数含义

运行下方单元，依据该 schema 定义你的工具。

In [7]:
# 手动定义工具 schema
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time", # <--- 你的函数名
        "description": "返回当前时间的字符串", # <--- 提供给 LLM 的描述
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}]

在这种手动定义 schema 的场景下，你需要自行处理执行流程。我们先定义响应以完成设置。

In [8]:
# 重新初始化消息
messages = [
    {
        "role": "user",
        "content": "现在几点？",
    }
]

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=tools
)

现在可以查看来自 LLM 的响应。

In [9]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "model": "glm-4.7-flash",
  "created": 1772093530,
  "choices": [
    {
      "index": 0,
      "finish_reason": "tool_calls",
      "message": {
        "content": "",
        "role": "assistant",
        "reasoning_content": "\u7528\u6237\u95ee\"\u73b0\u5728\u51e0\u70b9\uff1f\"\uff0c\u8fd9\u662f\u4e2d\u6587\uff0c\u610f\u601d\u662f\"What time is it now?\"\u3002\u6211\u9700\u8981\u4f7f\u7528get_current_time\u51fd\u6570\u6765\u83b7\u53d6\u5f53\u524d\u65f6\u95f4\u3002",
        "tool_calls": [
          {
            "id": "call_-7851652734842302538",
            "function": {
              "arguments": "{}",
              "name": "get_current_time"
            },
            "type": "function",
            "index": 0
          }
        ]
      }
    }
  ],
  "request_id": "20260226161209f04989f31c724aeb",
  "id": "20260226161209f04989f31c724aeb",
  "usage": {
    "prompt_tokens": 144,
    "prompt_tokens_details": {
      "cached_tokens": 43
    },
    "completion_tokens": 37,
    "

注意在 `response` 的 `message` 下可以看到 `tool_calls`。这表明 LLM 希望调用某个工具（此处为 `get_current_time`）。你可以添加一些逻辑来处理这一情况，然后将结果回传给模型并获得最终响应。

运行下方单元，在本地执行函数，返回结果给 LLM，并查看最终响应。

In [10]:
response2 = None

# 若响应对象中包含 tool_calls，则进入处理分支
if response.choices[0].message.tool_calls:
    # 从响应中提取具体的工具调用元数据
    tool_call = response.choices[0].message.tool_calls[0]
    if tool_call.function.arguments:
        args = json.loads(tool_call.function.arguments)
    else:
        args = {}

    # 在本地运行该工具
    tool_result = get_current_time()

    # 将工具结果追加到消息列表
    messages.append({
        "role": "assistant",
        "content": "",
        "tool_calls": [tool_call.model_dump()]
    })
    messages.append({
        "role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)
    })

    # 将包含新结果的消息列表回传给 LLM
    response2 = client.chat.completions.create(
        model="glm-4.7-flash",
        messages=messages,
        tools=tools,
    )

    print_html(f"最终响应: {response2.choices[0].message.content}")

至此，你实现了对 LLM 工具调用的手动处理。智谱AI SDK 需要你手动编写 schema 并手动处理中间步骤。

## 4. 为 LLM 提供更多工具

### 4.1 三个新工具

你将为 LLM 定义三个新工具：

- **Weather Tool (`get_weather_from_ip`)**
  通过外部 API 检测你的 IP 地址以获取地理位置，并调用天气 API 返回当前位置的当前温度、最高温与最低温。

- **File Writing Tool (`write_txt_file`)**
  在你的本地环境创建包含指定内容的文本文件。函数接收两个参数：`file_path` 与 `content`。

- **QR Code Generator (`generate_qr_code`)**
  根据数据生成二维码图像，可选嵌入中心图片。函数接收三个参数：`data`、`filename` 与 `img_path`。

运行下方单元以导入新包并定义这些工具。

In [11]:
import requests
import qrcode
from qrcode.image.styledpil import StyledPilImage


def get_weather_from_ip():
    """
    获取用户所在位置的当前、最高与最低温度（摄氏制），并返回给用户。
    """
    # 通过 IP 地址获取地理坐标
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')

    # 设置天气 API 调用的参数
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "celsius",
        "timezone": "auto"
    }

    # 获取天气数据
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # 格式化并返回简洁字符串
    return (
        f"Current: {weather_data['current']['temperature_2m']}°C, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°C, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°C"
    )

# 写入文本文件
def write_txt_file(file_path: str, content: str):
    """
    将字符串写入 .txt 文件（若已存在则覆盖）。
    参数：
        file_path (str)：目标路径。
        content (str)：要写入的文本。
    返回：
        str：写入文件的路径。
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path


# 生成二维码
def generate_qr_code(data: str, filename: str, image_path: str):
    """给定数据与图片路径，生成二维码图像。

    参数：
        data：要编码的文本或 URL
        filename：输出 PNG 文件名（不含扩展名）
        image_path：用于作为二维码背景的图片路径
    """
    from PIL import Image
    
    # 创建二维码
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)
    qr_img = qr.make_image(fill_color="black", back_color="white")
    
    # 打开背景图片
    background = Image.open(image_path).convert("RGBA")
    
    # 将二维码调整到与背景图片相同大小
    qr_img = qr_img.resize(background.size)
    
    # 将二维码转换为RGBA模式，设置透明度
    qr_img = qr_img.convert("RGBA")
    # 设置二维码的透明度（白色部分变透明）
    qr_data = qr_img.getdata()
    new_qr_data = []
    for item in qr_data:
        # 如果是白色，设为完全透明；如果是黑色，保持不透明
        if item[:3] == (255, 255, 255):
            new_qr_data.append((255, 255, 255, 0))  # 白色背景透明
        else:
            new_qr_data.append(item)  # 黑色像素保持原样
    qr_img.putdata(new_qr_data)
    
    # 将背景和二维码合成
    background.paste(qr_img, (0, 0), qr_img)
    
    output_file = f"{filename}.png"
    background.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

### 4.2 创建完整的工具列表

现在我们需要为智谱AI SDK 创建所有工具的 JSON Schema 定义。

In [12]:
# 定义所有工具的 JSON Schema
all_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "返回当前时间的字符串",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_from_ip",
            "description": "获取用户所在位置的当前、最高与最低温度（摄氏制）",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_txt_file",
            "description": "将字符串写入 .txt 文件（若已存在则覆盖）",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {
                        "type": "string",
                        "description": "目标文件路径"
                    },
                    "content": {
                        "type": "string",
                        "description": "要写入的文本内容"
                    }
                },
                "required": ["file_path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_qr_code",
            "description": "给定数据与图片路径，生成二维码图像",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "要编码的文本或 URL"
                    },
                    "filename": {
                        "type": "string",
                        "description": "输出 PNG 文件名（不含扩展名）"
                    },
                    "image_path": {
                        "type": "string",
                        "description": "用于嵌入到二维码中的图片路径"
                    }
                },
                "required": ["data", "filename", "image_path"]
            }
        }
    }
]

### 4.3 使用你的新工具

现在来使用这些新工具！`response` 的结构与之前类似，但不同的是你会把所有工具都传给 LLM。LLM 将基于你的提示选择合适的工具。我们先从 `get_weather_from_ip` 开始。

In [13]:
# 定义一个通用的工具处理函数
def handle_tool_calls_with_multiple_tools(response, messages, client, tools, max_iterations=5):
    """处理工具调用的循环，支持多个工具"""
    for iteration in range(max_iterations):
        if response.choices[0].message.tool_calls:
            tool_call = response.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            
            # 解析参数
            if tool_call.function.arguments:
                args = json.loads(tool_call.function.arguments)
            else:
                args = {}
            
            # 执行对应的工具函数
            if tool_name == "get_current_time":
                result = get_current_time()
            elif tool_name == "get_weather_from_ip":
                result = get_weather_from_ip()
            elif tool_name == "write_txt_file":
                result = write_txt_file(args.get("file_path"), args.get("content"))
            elif tool_name == "generate_qr_code":
                result = generate_qr_code(args.get("data"), args.get("filename"), args.get("image_path"))
            else:
                result = f"未知工具: {tool_name}"
            
            print(f"工具 {tool_name} 执行结果: {result}")
            
            # 将工具结果返回给LLM
            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": [tool_call.model_dump()]
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })
            
            # 获取下一轮响应
            response = client.chat.completions.create(
                model="glm-4.7-flash",
                messages=messages,
                tools=tools
            )
        else:
            break
    return response

# 测试天气查询
prompt = "可以帮我查询我所在位置的天气吗？"

messages = [
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=all_tools
)

# 处理工具调用
final_response = handle_tool_calls_with_multiple_tools(response, messages, client, all_tools, max_iterations=5)

print_html(f"\n最终响应: {final_response.choices[0].message.content}")

工具 get_weather_from_ip 执行结果: Current: 23.0°C, High: 23.0°C, Low: 19.9°C


你可以看到，即使 LLM 拥有其他工具的访问权限，它也能根据提示意图选择正确的工具。

现在运行下方单元，提示 LLM 为你创建一条备忘。

In [14]:
prompt = "请帮我创建一个名为 reminders.txt 的备忘录，提醒我明天晚上7点给 Daniel 打电话。"

messages = [
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=all_tools
)

final_response = handle_tool_calls_with_multiple_tools(response, messages, client, all_tools, max_iterations=5)

print(f"最终响应: {final_response.choices[0].message.content}")

工具 write_txt_file 执行结果: reminders.txt
最终响应: 已成功创建 `reminders.txt` 文件！文件中已包含提醒内容：**明天晚上7点给 Daniel 打电话**。

你可以：
1. 随时打开该文件查看提醒
2. 将文件放在容易访问的位置（如桌面、手机相册等）
3. 设置手机或电脑的闹钟作为备份提醒

祝你好运！


现在你已经拥有包含提醒内容的文本文件了。可以看到 LLM 为工具传递了正确的参数，工具在本地运行，并返回了已完成操作的响应。你甚至可以打开该文本文件检查其内容以确认文件已生成。

In [15]:
with open('reminders.txt', 'r') as file:
    contents = file.read()
    print(contents)

明天晚上7点给 Daniel 打电话


最后，使用二维码生成工具创建一个跳转到 DeepLearning.AI 网站的二维码。

运行下方单元以生成该二维码。

In [16]:
prompt = "请用我公司的 logo 生成一个指向 www.deeplearning.ai 的二维码；logo 位于 `567.png；文件名可命名为 dl_qr_code。"

messages = [
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=all_tools
)

final_response = handle_tool_calls_with_multiple_tools(response, messages, client, all_tools, max_iterations=5)

print(f"最终响应: {final_response.choices[0].message.content}")

/var/folders/xm/r_gmr5_118q_ysr_9xtd0klc0000gn/T/ipykernel_869/1627600530.py:73: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  qr_data = qr_img.getdata()


工具 generate_qr_code 执行结果: QR code saved as dl_qr_code.png containing: www.deeplearning.ai...
最终响应: 二维码已经成功生成！

**生成结果：**
- 文件名：`dl_qr_code.png`
- 内容：`www.deeplearning.ai`
- Logo：已嵌入 `567.png`

二维码文件已保存在当前目录下，您可以下载并使用它。


LLM 同样成功地将正确参数传给函数，随后使用这些信息运行函数并生成二维码。

### 4.4 同时使用多个工具

最后需要记住的是，LLM 可以按序列使用多个工具来完成多项任务。我们使用这些工具并查看 `response`，观察发生了什么。你将使用一个较复杂的提示：

> 你能帮我用图片 dl_logo.jpg 生成一个跳转到 www.deeplearning.com 的二维码吗？另外请写一个包含当前天气的 txt 备忘。

该提示需要一定的逻辑来决定何时调用哪些工具。例如，尽管你让它先写文本备忘并随后描述其内容，但 LLM 需要先获取天气信息才能写入备忘。如果它先调用 `write_txt_file`，则还没有来自 `get_weather_from_ip` 的信息。这个例子很好地展示了 LLM 解析自然语言并按正确顺序使用合适工具以完成多样任务的能力。

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">

<h4>🔍 注意要点：</h4>

<ul>
  <li>LLM 会根据用户请求<b>自动选择</b>合适的工具</li>
  <li>从用户消息中<b>自动推断参数</b>（如文件名、内容、URL）</li>
  <li>每个工具<b>返回信息</b>并被 LLM 纳入其最终响应</li>
  <li><b>无参数工具</b>（如天气与时间）非常适合快速信息查询</li>
  <li>尽管幕后有复杂操作，对话依然<b>自然</b></li>
</ul>

</div>

In [18]:
prompt = "能否帮我用图片 567.png 生成一个跳转到 www.deeplearning.com 的二维码？另外再写一个包含当前天气的 txt 备忘。"

messages = [
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=all_tools
)

final_response = handle_tool_calls_with_multiple_tools(response, messages, client, all_tools, max_iterations=10)

print(f"最终响应: {final_response.choices[0].message.content}")

/var/folders/xm/r_gmr5_118q_ysr_9xtd0klc0000gn/T/ipykernel_869/1627600530.py:73: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  qr_data = qr_img.getdata()


工具 generate_qr_code 执行结果: QR code saved as deeplearning_qr.png containing: www.deeplearning.com...
工具 get_weather_from_ip 执行结果: Current: 23.0°C, High: 23.0°C, Low: 19.9°C
工具 write_txt_file 执行结果: weather_backup.txt
最终响应: 我已经帮您完成了两项任务：

1. **生成二维码**：使用图片 `567.png` 作为嵌入图标，生成了一个跳转到 `www.deeplearning.com` 的二维码，保存为 `deeplearning_qr.png`。

2. **天气备份**：获取了当前天气信息并写入 `weather_backup.txt` 文件，内容包括：
   - 当前温度：23.0°C
   - 最高温度：23.0°C
   - 最低温度：19.9°C

文件已成功创建，您可以在当前目录下找到这两个文件！


可以从工具调用序列中看到，智能体按正确顺序调用了工具以完成任务，同时以你要求的顺序进行响应。

### 模型选项

在运行这些工具调用工作流时，你可以尝试不同的智谱AI模型。不同模型在能力、成本与速度之间提供不同平衡：

- **`glm-4.7-flash`** — 免费模型，优化了推理与速度，适合学习和开发
- **`glm-4`** — 强推理性能，适合复杂任务
- **`glm-4-air`** — 更轻、更快、且更便宜的模型

模型选择取决于你的目标：
- 小模型适合快速、低成本的原型开发
- 当任务需要更强推理或多步编排时，切换到更强的模型

### 关键收获

- 工具调用让 LLM 超越纯文本生成——它们可把函数作为推理的一部分。
- 清晰且文档完备的函数（精确的 docstring）有助于模型判断何时、如何使用工具。
- 智谱AI SDK 需要手动将 Python 函数转换为工具 schema 并手动编排多步工作流。
- 选择合适模型很重要：小模型在简单任务上更快更便宜，强模型更适合重推理的工作流。
- 观察完整的会话流程（提示、工具调用、结果、最终响应）对于调试与改进智能体行为至关重要。

理解以上要素后，你已具备将 LLM 推理与外部工具结合以完成更复杂任务的基础。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你已经完成了使用智谱AI SDK 将**函数变成工具**的实验。
在此过程中，<strong>你</strong>将 Python 函数暴露为工具，允许 LLM 选择并调用它们，并检查了多步工具编排。

具备这些技能后，<strong>你</strong>可以设计将 LLM 推理与实际动作结合的智能体工作流——可靠、可审计且易扩展。🌟

</div>